# Дообучение RuBERT для классификации сообщений

Эксперимент с fine-tuning трансформерной языковой модели для многоклассовой классификации сообщений; используются macro precision, recall, F1 и accuracy.

In [1]:
import pandas as pd
import json
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from imblearn.under_sampling import RandomUnderSampler

In [2]:
import warnings
warnings.filterwarnings("ignore")

In [3]:
seed = 42
np.random.seed(seed)

In [4]:
import re


def extract_text(msg):
    if not isinstance(msg, list):
        return msg

    result_text = ''
    for item in msg:
        if isinstance(item, str):
            result_text += (item + " ")
        else:
            result_text += (item['text'] + " ")
    return result_text

def prepoccess_dataset(df):
    preproccessed_df = df.copy()
    preproccessed_df = df.dropna(subset=['text'])
    preproccessed_df.reset_index(drop=True, inplace=True)
    preproccessed_df['text'] = preproccessed_df['text'].map(lambda x: extract_text(x))
    preproccessed_df['text_clean_nums'] = preproccessed_df['text'].map(lambda x: re.sub(r'\d', '', x))
    preproccessed_df = preproccessed_df[preproccessed_df['text_clean_nums'] != '']
    return preproccessed_df

def chat_to_df(json_path):
    with open(json_path, 'r', encoding='utf-8') as f:
        data_json = json.load(f)

    user_id = []
    usernames = []
    dates = []
    texts = []
    for msg in data_json['messages']:
        if msg['type'] == 'message':
            user_id.append(msg['from_id'])
            if 'forwarded_from' in msg.keys():
                usernames.append(msg['forwarded_from'])
            else:
                usernames.append(msg['from'])
            dates.append(msg['date'])
            texts.append(msg['text'])

    df = pd.DataFrame({
        'user_id': user_id,
        'username': usernames,
        'date': dates,
        'text': texts,
    })
    
    return prepoccess_dataset(df)

In [5]:
json_path = r'../../data/chats/marakasi_14.02.2026.json'
df = chat_to_df(json_path)

In [6]:
df.head()

,user_id,username,date,text,text_clean_nums
0,user674227701,свитистер МТС,2024-08-21T22:53:00,А мне,А мне
1,user790422015,Ири,2024-08-21T22:53:02,💋💋💋,💋💋💋
2,user790422015,Ири,2024-08-21T22:53:15,Пожалуйста не надо альфа Лера,Пожалуйста не надо альфа Лера
3,user872595910,ВАЛЬРА!,2024-08-21T22:53:20,Да,Да
4,user790422015,Ири,2024-08-21T22:53:23,А что ты хочешь,А что ты хочешь


In [7]:
df = df.drop(columns=['user_id', 'date'])

In [8]:
marakasi_users = [
    'свитистер МТС', 'Ири', 'ВАЛЬРА!', 'Шорт', 'Арина Колошек🌏',
    'Соня🤘', 'Егорчек🙄', 'Denis ss', 'dandelions', 'Умка', 'nuta',
    '見知らぬ人 アイ', 'Кибержаба'
]

marakasi_users = [
    'свитистер МТС', 'Ири', 'ВАЛЬРА!', 'Шорт', 'Арина Колошек🌏',
    'Соня🤘', 'Егорчек🙄', 'Denis ss', 'dandelions', 'Умка', 'nuta',
]

In [9]:
df = df[df['username'].isin(marakasi_users)]

In [10]:
df['username'].value_counts()

username
dandelions        16662
свитистер МТС     10964
ВАЛЬРА!           10634
Соня🤘              4487
Шорт               4269
Егорчек🙄           3695
Ири                2461
Denis ss           2387
Арина Колошек🌏     2268
Умка               2161
nuta               2023
Name: count, dtype: int64

In [11]:
#df, _ = train_test_split(df, stratify=df['username'], train_size=0.2)

# rus = RandomUnderSampler(random_state=42)
# X_df = df.drop(columns=['username'])
# y_df = df[['username']]
# X_resample, y_resample = rus.fit_resample(X_df, y_df)
# df = pd.concat([X_resample, y_resample], axis=1)

In [12]:
df['username'].value_counts()

username
dandelions        16662
свитистер МТС     10964
ВАЛЬРА!           10634
Соня🤘              4487
Шорт               4269
Егорчек🙄           3695
Ири                2461
Denis ss           2387
Арина Колошек🌏     2268
Умка               2161
nuta               2023
Name: count, dtype: int64

In [13]:
from sklearn.preprocessing import LabelEncoder

In [14]:
le = LabelEncoder()
df['label'] = le.fit_transform(df['username'])

In [15]:
id2label = {int(i): str(label) for i, label in enumerate(le.classes_)}
label2id = {str(label): int(i) for i, label in enumerate(le.classes_)}

In [16]:
id2label

{0: 'Denis ss',
 1: 'dandelions',
 2: 'nuta',
 3: 'Арина Колошек🌏',
 4: 'ВАЛЬРА!',
 5: 'Егорчек🙄',
 6: 'Ири',
 7: 'Соня🤘',
 8: 'Умка',
 9: 'Шорт',
 10: 'свитистер МТС'}

In [17]:
df = df.drop(columns=['text'])

In [18]:
df.head()

,username,text_clean_nums,label
0,свитистер МТС,А мне,10
1,Ири,💋💋💋,6
2,Ири,Пожалуйста не надо альфа Лера,6
3,ВАЛЬРА!,Да,4
4,Ири,А что ты хочешь,6


In [19]:
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

W0329 19:01:51.590000 25392 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [20]:
model_id = 'DeepPavlov/rubert-base-cased-conversational'
tokenizer = AutoTokenizer.from_pretrained(model_id)

In [21]:
dataset = Dataset.from_pandas(df[['text_clean_nums', 'label']])
dataset = dataset.class_encode_column("label")
dataset = dataset.train_test_split(test_size=0.2, stratify_by_column="label", seed=42)

Stringifying the column:   0%|          | 0/62011 [00:00<?, ? examples/s]

Casting to class labels:   0%|          | 0/62011 [00:00<?, ? examples/s]

In [22]:
def tokenize_func(example):
    return tokenizer(example['text_clean_nums'], padding='max_length', max_length=128, truncation=True)

tokenized_dataset = dataset.map(tokenize_func, batched=True)

Map:   0%|          | 0/49608 [00:00<?, ? examples/s]

Map:   0%|          | 0/12403 [00:00<?, ? examples/s]

In [23]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_id,
    num_labels=len(df['label'].unique()),
    id2label=id2label, label2id=label2id,
    device_map='auto', dtype='auto'
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at DeepPavlov/rubert-base-cased-conversational and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [24]:
for param in model.base_model.parameters():
    param.requires_grad = False

for name, param in model.named_parameters():
    if param.requires_grad:
        print(f"Обучаемый слой: {name}")

Обучаемый слой: classifier.weight
Обучаемый слой: classifier.bias


In [25]:
from sklearn.utils.class_weight import compute_class_weight

weights = compute_class_weight(
    class_weight='balanced', 
    classes=np.unique(dataset['train']['label']), 
    y=dataset['train']['label']
)

In [26]:
np.unique(dataset['train']['label'])

array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10])

In [27]:
weights

array([2.36116135, 0.33834633, 0.51417378, 2.78727947, 2.48611807,
       0.53013027, 1.52564891, 2.29041045, 1.25621676, 2.60833903,
       1.32059098])

In [28]:
id2label

{0: 'Denis ss',
 1: 'dandelions',
 2: 'nuta',
 3: 'Арина Колошек🌏',
 4: 'ВАЛЬРА!',
 5: 'Егорчек🙄',
 6: 'Ири',
 7: 'Соня🤘',
 8: 'Умка',
 9: 'Шорт',
 10: 'свитистер МТС'}

In [29]:
from torch import nn
import torch

In [30]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else 'cpu'

In [31]:
device

'cuda'

In [32]:
import numpy as np
import evaluate

def compute_metrics(eval_pred):
    precision_metric = evaluate.load("precision")
    recall_metric = evaluate.load("recall")
    f1_metric = evaluate.load("f1")
    accuracy_metric = evaluate.load("accuracy")
    
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    
    results = {}
    results.update(precision_metric.compute(predictions=predictions, references=labels, average='macro'))
    results.update(recall_metric.compute(predictions=predictions, references=labels, average='macro'))
    results.update(f1_metric.compute(predictions=predictions, references=labels, average='macro'))
    results.update(accuracy_metric.compute(predictions=predictions, references=labels))
    
    return results

In [33]:
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        
        loss_fct = nn.CrossEntropyLoss(weight=torch.tensor(weights).float().to(device))
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        
        return (loss, outputs) if return_outputs else loss

In [35]:
training_args = TrainingArguments(
    output_dir="../../models/marakasi_rubert",
    learning_rate=5e-5,
    lr_scheduler_type="cosine",
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    fp16=True,
    num_train_epochs=15,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="no",
    load_best_model_at_end=False,
    logging_strategy='epoch'
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    compute_metrics=compute_metrics
)

trainer.train()

The model is already on multiple devices. Skipping the move to device specified in `args`.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\Denis\_netrc.
wandb: Currently logged in as: salnikovdenis44 (salnikovdenis44-omstu) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


--- Logging error ---
Traceback (most recent call last):
  File "C:\Users\Denis\miniconda3\envs\myenv\lib\logging\__init__.py", line 1103, in emit
    stream.write(msg + self.terminator)
  File "C:\Users\Denis\miniconda3\envs\myenv\lib\encodings\cp1251.py", line 19, in encode
    return codecs.charmap_encode(input,self.errors,encoding_table)[0]
UnicodeEncodeError: 'charmap' codec can't encode character '\U0001f30f' in position 554: character maps to <undefined>
Call stack:
  File "C:\Users\Denis\miniconda3\envs\myenv\lib\runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "C:\Users\Denis\miniconda3\envs\myenv\lib\runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "C:\Users\Denis\miniconda3\envs\myenv\lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\Denis\miniconda3\envs\myenv\lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,2.375700,2.344963,0.164326,0.139925,0.126655,0.235911
2,2.341000,2.313754,0.163030,0.161556,0.144372,0.258002
3,2.314600,2.287670,0.184232,0.184044,0.166872,0.243489
4,2.293700,2.267007,0.189634,0.186101,0.149655,0.192937
5,2.279000,2.250968,0.194136,0.195764,0.167151,0.239297
6,2.264000,2.238278,0.198617,0.204011,0.170654,0.232444
7,2.251600,2.228380,0.204767,0.207804,0.167976,0.226639
8,2.243300,2.220635,0.201966,0.210619,0.180253,0.248408
9,2.238600,2.214660,0.208974,0.208264,0.187180,0.280174
10,2.231700,2.210393,0.201244,0.211451,0.189071,0.264613


TrainOutput(global_step=23265, training_loss=2.2642483994701, metrics={'train_runtime': 2292.7678, 'train_samples_per_second': 324.551, 'train_steps_per_second': 10.147, 'total_flos': 4.895050487049216e+16, 'train_loss': 2.2642483994701, 'epoch': 15.0})

In [52]:
trainer.save_model("../../models/marakasi_final_model")

In [40]:
from transformers import pipeline

In [75]:
path = r"../../models/marakasi_final_model"

model = AutoModelForSequenceClassification.from_pretrained(
    path,
    device_map='auto', dtype='auto'
)

In [76]:
classifier = pipeline('sentiment-analysis', model=model, tokenizer=tokenizer, device_map='auto')

Device set to use cuda:0


In [80]:
classifier("моя мечта сука")

[{'label': 'dandelions', 'score': 0.2715003192424774}]